In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report
import numpy as np

In [2]:
#Similarly, you'll need to change the file location to match your own for this to work
df = pd.read_csv('C:/Users/zitin/Documents/BAC Datathon/Moomoo and Friends_Finals/Mobile Reviews Sentiment.csv')
df.head()

,review_id,customer_name,age,brand,model,price_usd,exchange_rate_to_usd,rating,review_text,sentiment,...,verified_purchase,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,review_length,word_count,helpful_votes,source
0,1,Aryan Maharaj,45,Realme,Realme 12 Pro,337.31,83.00,2,Not worth the money spent. Wouldn’t recommend.,Negative,...,True,1,1,3,2,1,46,7,1,Amazon
1,2,Davi Miguel Sousa,18,Realme,Realme 12 Pro,307.78,5.70,4,Absolutely love this phone! The camera is next...,Positive,...,True,3,2,4,3,2,74,12,5,Flipkart
2,3,Pahal Balay,27,Google,Pixel 6,864.53,83.00,4,Loving the clean UI and fast updates. Loving i...,Positive,...,True,3,5,3,2,4,55,11,8,AliExpress
3,4,David Guzman,19,Xiaomi,Redmi Note 13,660.94,3.67,3,Build quality feels solid and durable. No regr...,Positive,...,False,1,3,2,1,2,66,11,3,Amazon
4,5,Yago Leão,38,Motorola,Edge 50,792.13,5.70,3,Not bad for daily use but could be optimized. ...,Neutral,...,True,3,3,2,2,1,73,12,0,BestBuy


In [3]:
print("Data type : ", type(df))
print("Data dims : ", df.shape)
print(df.dtypes)

Data type :  <class 'pandas.core.frame.DataFrame'>
Data dims :  (50000, 23)
review_id                 int64
customer_name            object
age                       int64
brand                    object
model                    object
price_usd               float64
exchange_rate_to_usd    float64
rating                    int64
review_text              object
sentiment                object
country                  object
language                 object
review_date              object
verified_purchase          bool
battery_life_rating       int64
camera_rating             int64
performance_rating        int64
design_rating             int64
display_rating            int64
review_length             int64
word_count                int64
helpful_votes             int64
source                   object
dtype: object


In [6]:
# Step 1: Prepare Data
df['sentiment_binary'] = df['sentiment'].map({'Positive': 1, 'Negative': -1, 'Neutral': 0})

# Step 2: Split Data
X = df[['review_text']]
y = df['sentiment_binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Text Vectorization
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

# Step 4: Pipeline
pipeline = Pipeline([
    ('vectorize', tfidf),
    ('clf', MultinomialNB())
])

# Step 5: Fit Model
pipeline.fit(X_train['review_text'], y_train)

# Step 6: Evaluate
y_pred = pipeline.predict(X_test['review_text'])
print(classification_report(y_test, y_pred))

# Step 7: Extract Top Words
feature_names = pipeline.named_steps['vectorize'].get_feature_names_out()
log_probs = pipeline.named_steps['clf'].feature_log_prob_

top_pos = np.argsort(log_probs[1])[-10:]
top_neut = np.argsort(log_probs[0])[-10:]
top_neut = np.argsort(log_probs[-1])[-10:]

print("Top Positive Words:", feature_names[top_pos])
print("Top Negative Words:", feature_names[top_neg])
print("Top Neutral Words:", feature_names[top_neut])


              precision    recall  f1-score   support

          -1       1.00      1.00      1.00      1998
           0       1.00      1.00      1.00      2538
           1       1.00      1.00      1.00      5464

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000

Top Positive Words: ['bad' 'bit' 'experience' 'use' 'casual' 'overall' 'average' 'better'
 'fine' 'okay']
Top Negative Words: ['spent' 'money' 'quality' 'phone' 'returning' 'soon' 'wouldn' 'recommend'
 'mark' 'disappointed']
Top Neutral Words: ['smooth' 'best' 'year' 'purchase' 'buying' 'regrets' 'far' 'worth'
 'absolutely' 'loving']
